<a href="https://colab.research.google.com/github/the-name-is-madhu/BIS_LAB/blob/main/1BM24CS154__Financial_Portfolio_Optimization__Grey_Wolf_for_Optimization_Problem_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import numpy as np
import pandas as pd

# ==========================================
# 1. GENERATE HISTORICAL ASSET RETURNS
# ==========================================

np.random.seed(42)

assets = [
    "Stock_A",
    "Stock_B",
    "Stock_C",
    "Stock_D",
    "Stock_E"
]

# Generate sample daily historical returns
returns = pd.DataFrame({
    "Stock_A": np.random.normal(0.0008, 0.015, 500),
    "Stock_B": np.random.normal(0.0006, 0.012, 500),
    "Stock_C": np.random.normal(0.0009, 0.020, 500),
    "Stock_D": np.random.normal(0.0005, 0.010, 500),
    "Stock_E": np.random.normal(0.0010, 0.018, 500)
})

# Calculate mean returns and covariance matrix
mean_returns = returns.mean().values
cov_matrix = returns.cov().values

n = len(assets)

# Investment budget
budget = 100000

# Maximum allocation per asset
max_weight = 0.60

# Risk aversion parameter
risk_lambda = 2.0


# ==========================================
# 2. GREY WOLF OPTIMIZER PARAMETERS
# ==========================================

n_wolves = 25
max_iterations = 200


# ==========================================
# 3. NORMALIZE PORTFOLIO WEIGHTS
# ==========================================

def normalize_weights(weights):

    weights = np.asarray(weights, dtype=float)

    # Project weights onto constrained simplex
    low = np.min(weights) - max_weight
    high = np.max(weights)

    for _ in range(100):

        mid = (low + high) / 2

        projected = np.clip(
            weights - mid,
            0,
            max_weight
        )

        if projected.sum() > 1:
            low = mid
        else:
            high = mid

    result = np.clip(
        weights - (low + high) / 2,
        0,
        max_weight
    )

    return result / result.sum()


# ==========================================
# 4. FITNESS FUNCTION
# ==========================================

def fitness(weights):

    # Calculate expected portfolio return
    portfolio_return = np.dot(
        weights,
        mean_returns
    )

    # Calculate portfolio variance
    portfolio_variance = np.dot(
        weights,
        np.dot(cov_matrix, weights)
    )

    # Risk-adjusted objective
    score = (
        portfolio_return
        - risk_lambda * portfolio_variance
    )

    return score


# ==========================================
# 5. INITIALIZE WOLF POPULATION
# ==========================================

wolves = np.array([
    normalize_weights(np.random.rand(n))
    for _ in range(n_wolves)
])

# Initialize alpha, beta and delta
alpha_wolf = None
beta_wolf = None
delta_wolf = None

alpha_score = -np.inf
beta_score = -np.inf
delta_score = -np.inf


# ==========================================
# 6. MAIN GREY WOLF OPTIMIZER
# ==========================================

for iteration in range(max_iterations):

    # Reset hierarchy for current population
    alpha_score = -np.inf
    beta_score = -np.inf
    delta_score = -np.inf

    # Evaluate all wolves
    for i in range(n_wolves):

        score = fitness(wolves[i])

        if score > alpha_score:

            delta_score = beta_score
            delta_wolf = (
                beta_wolf.copy()
                if beta_wolf is not None else None
            )

            beta_score = alpha_score
            beta_wolf = (
                alpha_wolf.copy()
                if alpha_wolf is not None else None
            )

            alpha_score = score
            alpha_wolf = wolves[i].copy()

        elif score > beta_score:

            delta_score = beta_score
            delta_wolf = (
                beta_wolf.copy()
                if beta_wolf is not None else None
            )

            beta_score = score
            beta_wolf = wolves[i].copy()

        elif score > delta_score:

            delta_score = score
            delta_wolf = wolves[i].copy()

    # Parameter a decreases linearly from 2 to 0
    a = 2 - iteration * (2 / max_iterations)

    # Update positions of all wolves
    for i in range(n_wolves):

        X = wolves[i].copy()

        # Alpha wolf update
        r1 = np.random.rand(n)
        r2 = np.random.rand(n)

        A1 = 2 * a * r1 - a
        C1 = 2 * r2

        D_alpha = np.abs(
            C1 * alpha_wolf - X
        )

        X1 = alpha_wolf - A1 * D_alpha

        # Beta wolf update
        r1 = np.random.rand(n)
        r2 = np.random.rand(n)

        A2 = 2 * a * r1 - a
        C2 = 2 * r2

        D_beta = np.abs(
            C2 * beta_wolf - X
        )

        X2 = beta_wolf - A2 * D_beta

        # Delta wolf update
        r1 = np.random.rand(n)
        r2 = np.random.rand(n)

        A3 = 2 * a * r1 - a
        C3 = 2 * r2

        D_delta = np.abs(
            C3 * delta_wolf - X
        )

        X3 = delta_wolf - A3 * D_delta

        # Calculate new wolf position
        new_position = (
            X1 + X2 + X3
        ) / 3

        # Apply portfolio constraints
        wolves[i] = normalize_weights(
            new_position
        )

    # Display progress
    if (iteration + 1) % 50 == 0:

        print(
            "Iteration:", iteration + 1,
            "| Best Fitness:",
            round(alpha_score, 8)
        )


# ==========================================
# 7. OPTIMAL PORTFOLIO CALCULATION
# ==========================================

optimal_weights = alpha_wolf

portfolio_return = np.dot(
    optimal_weights,
    mean_returns
)

portfolio_variance = np.dot(
    optimal_weights,
    np.dot(cov_matrix, optimal_weights)
)

portfolio_risk = np.sqrt(
    portfolio_variance
)

# Calculate investment allocation
investment = optimal_weights * budget


# ==========================================
# 8. DISPLAY OPTIMAL PORTFOLIO
# ==========================================

print("\n======================================")
print("     OPTIMAL PORTFOLIO USING GWO")
print("======================================")

result = pd.DataFrame({

    "Asset": assets,

    "Optimal Weight (%)":
        optimal_weights * 100,

    "Investment Amount (Rs.)":
        investment

})

print(
    result.to_string(
        index=False,
        formatters={
            "Optimal Weight (%)": "{:.2f}".format,
            "Investment Amount (Rs.)": "{:.2f}".format
        }
    )
)


# ==========================================
# 9. DISPLAY PERFORMANCE METRICS
# ==========================================

print("\n======================================")
print("          PERFORMANCE METRICS")
print("======================================")

print(
    "Total Investment: Rs.",
    round(investment.sum(), 2)
)

print(
    "Expected Daily Return:",
    round(portfolio_return, 6)
)

print(
    "Daily Portfolio Variance:",
    round(portfolio_variance, 8)
)

print(
    "Daily Portfolio Risk:",
    round(portfolio_risk, 6)
)

print(
    "Annualized Expected Return (%):",
    round(portfolio_return * 252 * 100, 2)
)

print(
    "Annualized Portfolio Risk (%):",
    round(portfolio_risk * np.sqrt(252) * 100, 2)
)

print(
    "Best Fitness Score:",
    round(fitness(optimal_weights), 8)
)


# ==========================================
# 10. CONSTRAINT VERIFICATION
# ==========================================

print("\n======================================")
print("       CONSTRAINT VERIFICATION")
print("======================================")

print(
    "Budget Constraint:",
    "Satisfied"
    if abs(optimal_weights.sum() - 1) < 1e-6
    else "Not Satisfied"
)

print(
    "Maximum Allocation Constraint:",
    "Satisfied"
    if np.all(optimal_weights <= max_weight + 1e-6)
    else "Not Satisfied"
)

print(
    "Non-negative Weight Constraint:",
    "Satisfied"
    if np.all(optimal_weights >= 0)
    else "Not Satisfied"
)

print("\nGWO Optimization Completed Successfully!")


Iteration: 50 | Best Fitness: 0.00189944
Iteration: 100 | Best Fitness: 0.00189948
Iteration: 150 | Best Fitness: 0.00189958
Iteration: 200 | Best Fitness: 0.0018996

     OPTIMAL PORTFOLIO USING GWO
  Asset Optimal Weight (%) Investment Amount (Rs.)
Stock_A              14.77                14770.96
Stock_B              21.86                21855.62
Stock_C              60.00                60000.00
Stock_D               3.22                 3218.62
Stock_E               0.15                  154.80

          PERFORMANCE METRICS
Total Investment: Rs. 100000.0
Expected Daily Return: 0.002218
Daily Portfolio Variance: 0.00015907
Daily Portfolio Risk: 0.012612
Annualized Expected Return (%): 55.89
Annualized Portfolio Risk (%): 20.02
Best Fitness Score: 0.0018996

       CONSTRAINT VERIFICATION
Budget Constraint: Satisfied
Maximum Allocation Constraint: Satisfied
Non-negative Weight Constraint: Satisfied

GWO Optimization Completed Successfully!
